In [12]:
from pyspark.sql import SparkSession

# Create a spark session (which will run spark jobs)
spark = (
    SparkSession.builder.appName("MAST30034 Tutorial 1")
    .config("spark.sql.repl.eagerEval.enabled", True) 
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .getOrCreate()
)

In [13]:
import pandas as pd

# data output directory is
output_relative_dir = '../../data/'
abs_output_dir = output_relative_dir + 'raw_abs'

# Reading in the POA <-> SA2 mapping dataset produced in the previous run 
mb_poa_merged = pd.read_parquet(f"{abs_output_dir}/mb_poa_merged.parquet")

In [14]:
mb_poa_merged.head()

,MB_CODE_2021,SA2_CODE_2021,SA2_NAME_2021,POA_CODE_2021,POA_NAME_2021
0,10000009499,199999499,No usual address (NSW),9494,No usual address (Aust.)
1,10000010000,109011172,Albury - East,2640,2640
2,10000021000,109011176,Lavington,2641,2641
3,10000022000,109011176,Lavington,2641,2641
4,10000023000,109011176,Lavington,2641,2641


In [15]:
# mb_poa_merged is the merged mesh-block-level table with POA_CODE_2021 and SA2_CODE_2021

# Count the number of mesh blocks that falls within the unique post code + SA2 pairs
counts = mb_poa_merged.groupby(['POA_CODE_2021', 'SA2_CODE_2021']).size().reset_index(name='mb_count')

# Sort the df such that the rows are arranged with postcodes sorted in ascending order
# mb_count is sorted from highest to lowest
counts = counts.sort_values(['POA_CODE_2021', 'mb_count'], ascending=[True, False])
print(counts.shape)
print(counts.info())

# if duplicates are present, for example, the rows with the same postcode, keep the first one (the highest mb_count)
# and remove the others 
poa_to_sa2 = counts.drop_duplicates(subset='POA_CODE_2021', keep='first')
print(poa_to_sa2.shape)
print(poa_to_sa2.info())

# Producing the ratio of, in every mesh block of one particular post code, how many of blocks, fall inside a particular SA2? 
poa_totals = mb_poa_merged.groupby('POA_CODE_2021').size().reset_index(name='poa_total')
poa_to_sa2 = poa_to_sa2.merge(poa_totals, on='POA_CODE_2021')
poa_to_sa2['ratio'] = poa_to_sa2['mb_count'] / poa_to_sa2['poa_total']
print(poa_to_sa2.shape)
print(poa_to_sa2.head(10))


(5904, 3)
<class 'pandas.core.frame.DataFrame'>
Index: 5904 entries, 0 to 5903
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   POA_CODE_2021  5904 non-null   object
 1   SA2_CODE_2021  5904 non-null   object
 2   mb_count       5904 non-null   int64 
dtypes: int64(1), object(2)
memory usage: 184.5+ KB
None
(2644, 3)
<class 'pandas.core.frame.DataFrame'>
Index: 2644 entries, 0 to 5903
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   POA_CODE_2021  2644 non-null   object
 1   SA2_CODE_2021  2644 non-null   object
 2   mb_count       2644 non-null   int64 
dtypes: int64(1), object(2)
memory usage: 82.6+ KB
None
(2644, 5)
  POA_CODE_2021 SA2_CODE_2021  mb_count  poa_total     ratio
0          0800     701011002        93         93  1.000000
1          0810     701021025        69        459  0.150327
2          0812     701021022        74   

In [16]:
poa_to_sa2['POA_CODE_2021'].duplicated().sum()

np.int64(0)

In [17]:
poa_to_sa2.isna().sum()

POA_CODE_2021    0
SA2_CODE_2021    0
mb_count         0
poa_total        0
ratio            0
dtype: int64

In [19]:
# data output directory is
output_relative_dir = '../../data/'
abs_output_dir = output_relative_dir + 'raw_abs'

# save the cleaned POA <-> SA2 mapping df to be reused in the next step 
poa_to_sa2.to_parquet(f"{abs_output_dir}/poa_to_sa2.parquet", index=False)

26/09/19 01:52:41 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 1041092 ms exceeds timeout 120000 ms
26/09/19 01:52:41 WARN SparkContext: Killing executors is not supported by current scheduler.
26/09/19 01:55:02 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:70)
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:44)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:34)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.stor